In [1]:
"""
Build assumption_master sheet: ci_table_mapping
-----
Maps each CI product option to its rating table names and metadata.
Consumed by CI pricing rows 33-45 via CI_Product_Mapping named range.

Schema (wide): product_option | accelerator_table | standalone_table
             | age_definition | variant_type
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "ci_table_mapping"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: ci_table_mapping


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

# Locate Table Mappings anchor
hits = find_all_cells_any_col(ws, "Table Mappings")
print(f"'Table Mappings' matches: {hits}")

# Preview rows around it
for r, c in hits:
    print(f"\nRow {r} (col {get_col_letter(c)}) + 8 rows below, cols {get_col_letter(c)}-{get_col_letter(c+4)}:")
    print("-" * 90)
    for row in range(r, r + 9):
        vals = []
        for col_off in range(0, 5):
            v = ws.cell(row=row, column=c + col_off).value
            vals.append(f"{get_col_letter(c + col_off)}{row}={v!r}")
        print(f"  {' | '.join(vals)}")

'Table Mappings' matches: [(325, 5)]

Row 325 (col E) + 8 rows below, cols E-I:
------------------------------------------------------------------------------------------
  E325='Table Mappings' | F325='Accelerator' | G325='Standalone' | H325='Age Definition' | I325='Type'
  E326='Cancer_Protect' | F326='CI_Cancer_Protect' | G326='CI_Cancer_Protect' | H326='ALB' | I326='New'
  E327='Core' | F327='CI_Core_Accelerator' | G327='CI_Core_Standalone' | H327='ANB' | I327='Existing'
  E328='Essential' | F328='CI_Essential' | G328='CI_Essential' | H328='ALB' | I328='New'
  E329='Comprehensive' | F329='CI_Comprehensive_Accelerator' | G329='CI_Comprehensive_Standalone' | H329='ANB' | I329='Existing'
  E330='Comprehensive_Plus' | F330='CI_Comprehensive_Plus' | G330='CI_Comprehensive_Plus' | H330='ALB' | I330='New'
  E331='Extensive' | F331='CI_Extensive_Accelerator' | G331='CI_Extensive_Standalone' | H331='ANB' | I331='Existing'
  E332='Extensive_Plus' | F332='CI_Extensive_Plus' | G332='CI_Extensi

In [3]:
# =============================================================================
# CI Table Mappings — 7 product options × 4 attributes
# Layout: E=product_option, F=accelerator, G=standalone, H=age_def, I=type
# Source rows: 326-332 (7 rows)
# =============================================================================

TABLE_ROW_START = 326
TABLE_ROW_END   = 333  # exclusive

all_rows: list[dict] = []

for row in range(TABLE_ROW_START, TABLE_ROW_END):
    product_option    = ws.cell(row=row, column=5).value  # E
    accelerator_table = ws.cell(row=row, column=6).value  # F
    standalone_table  = ws.cell(row=row, column=7).value  # G
    age_definition    = ws.cell(row=row, column=8).value  # H
    variant_type      = ws.cell(row=row, column=9).value  # I

    # Structural check: all 5 cells must have values
    if product_option is None:
        raise RuntimeError(f"Empty product_option at row {row}")
    for label, val in [
        ("accelerator_table", accelerator_table),
        ("standalone_table",  standalone_table),
        ("age_definition",    age_definition),
        ("variant_type",      variant_type),
    ]:
        if val is None:
            raise RuntimeError(f"Empty {label} at row {row} (product={product_option})")

    all_rows.append({
        "product_option":    product_option,
        "accelerator_table": accelerator_table,
        "standalone_table":  standalone_table,
        "age_definition":    age_definition,
        "variant_type":      variant_type,
        "source_row":        row,
        "notes":             f"CI table mapping for {product_option}",
    })

print(f"Total rows harvested: {len(all_rows)}")
print()
for r in all_rows:
    print(f"  {r['product_option']:20s} | acc={r['accelerator_table']:35s} | "
          f"std={r['standalone_table']:35s} | {r['age_definition']:3s} | {r['variant_type']}")

Total rows harvested: 7

  Cancer_Protect       | acc=CI_Cancer_Protect                   | std=CI_Cancer_Protect                   | ALB | New
  Core                 | acc=CI_Core_Accelerator                 | std=CI_Core_Standalone                  | ANB | Existing
  Essential            | acc=CI_Essential                        | std=CI_Essential                        | ALB | New
  Comprehensive        | acc=CI_Comprehensive_Accelerator        | std=CI_Comprehensive_Standalone         | ANB | Existing
  Comprehensive_Plus   | acc=CI_Comprehensive_Plus               | std=CI_Comprehensive_Plus               | ALB | New
  Extensive            | acc=CI_Extensive_Accelerator            | std=CI_Extensive_Standalone             | ANB | Existing
  Extensive_Plus       | acc=CI_Extensive_Plus                   | std=CI_Extensive_Plus                   | ALB | New


In [4]:
df = pl.DataFrame(all_rows).select([
    "product_option",
    "accelerator_table",
    "standalone_table",
    "age_definition",
    "variant_type",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
assert df["product_option"].n_unique() == df.height, "Duplicate product_option"
assert df.height == 7, f"Expected 7 CI products, got {df.height}"

# Expected product options
expected_products = {
    "Cancer_Protect", "Core", "Essential", "Comprehensive",
    "Comprehensive_Plus", "Extensive", "Extensive_Plus",
}
actual_products = set(df["product_option"].to_list())
missing = expected_products - actual_products
unexpected = actual_products - expected_products
assert not missing, f"Missing: {missing}"
assert not unexpected, f"Unexpected: {unexpected}"

# Age definition must be ALB or ANB
allowed_age_def = {"ALB", "ANB"}
actual_age_defs = set(df["age_definition"].to_list())
assert actual_age_defs.issubset(allowed_age_def), (
    f"Unexpected age_definition values: {actual_age_defs - allowed_age_def}"
)

# Variant type must be New or Existing
allowed_types = {"New", "Existing"}
actual_types = set(df["variant_type"].to_list())
assert actual_types.issubset(allowed_types), (
    f"Unexpected variant_type: {actual_types - allowed_types}"
)

# =============================================================================
# Business rule cross-check: ALB↔New and ANB↔Existing
# From the source and your handover, ALB products are New CI generation,
# ANB products are Existing. Verify the pattern holds.
# =============================================================================
for row in df.iter_rows(named=True):
    age_def = row["age_definition"]
    variant = row["variant_type"]
    is_alb_new  = (age_def == "ALB" and variant == "New")
    is_anb_exist = (age_def == "ANB" and variant == "Existing")
    assert is_alb_new or is_anb_exist, (
        f"{row['product_option']}: unexpected combo age_def={age_def} × "
        f"variant={variant}. Expected ALB+New or ANB+Existing."
    )

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_row(product):
    row = df.filter(pl.col("product_option") == product)
    assert row.height == 1, f"Missing product {product}"
    return row.to_dicts()[0]

frozen = [
    ("Cancer_Protect",     "CI_Cancer_Protect",            "CI_Cancer_Protect",             "ALB", "New"),
    ("Core",               "CI_Core_Accelerator",          "CI_Core_Standalone",            "ANB", "Existing"),
    ("Essential",          "CI_Essential",                 "CI_Essential",                  "ALB", "New"),
    ("Comprehensive",      "CI_Comprehensive_Accelerator", "CI_Comprehensive_Standalone",   "ANB", "Existing"),
    ("Comprehensive_Plus", "CI_Comprehensive_Plus",        "CI_Comprehensive_Plus",         "ALB", "New"),
    ("Extensive",          "CI_Extensive_Accelerator",     "CI_Extensive_Standalone",       "ANB", "Existing"),
    ("Extensive_Plus",     "CI_Extensive_Plus",            "CI_Extensive_Plus",             "ALB", "New"),
]
for product, exp_acc, exp_std, exp_age, exp_var in frozen:
    row = get_row(product)
    assert row["accelerator_table"] == exp_acc, f"{product} acc: got {row['accelerator_table']}"
    assert row["standalone_table"]  == exp_std, f"{product} std: got {row['standalone_table']}"
    assert row["age_definition"]    == exp_age, f"{product} age: got {row['age_definition']}"
    assert row["variant_type"]      == exp_var, f"{product} var: got {row['variant_type']}"

# =============================================================================
# Interesting invariant: New products have same acc==std table (single table)
# Existing products have DIFFERENT acc and std tables (separate accelerator/standalone).
# =============================================================================
for row in df.iter_rows(named=True):
    is_new = row["variant_type"] == "New"
    same_table = row["accelerator_table"] == row["standalone_table"]
    if is_new:
        assert same_table, (
            f"{row['product_option']}: New variant should have same acc/std table, "
            f"but got acc={row['accelerator_table']}, std={row['standalone_table']}"
        )
    else:
        assert not same_table, (
            f"{row['product_option']}: Existing variant should have DIFFERENT "
            f"acc/std tables, but both are {row['accelerator_table']}"
        )

print(f"✓ {df.height} rows validated")
print(f"✓ 7 product options present")
print(f"✓ Age definitions: {actual_age_defs}")
print(f"✓ Variant types: {actual_types}")
print(f"✓ ALB↔New / ANB↔Existing invariant holds for all rows")
print(f"✓ New products have single table; Existing products have separate acc/std tables")
print(f"✓ {len(frozen)} frozen value checks passed")

print(f"\nFinal DataFrame:")
print(df)

✓ 7 rows validated
✓ 7 product options present
✓ Age definitions: {'ALB', 'ANB'}
✓ Variant types: {'Existing', 'New'}
✓ ALB↔New / ANB↔Existing invariant holds for all rows
✓ New products have single table; Existing products have separate acc/std tables
✓ 7 frozen value checks passed

Final DataFrame:
shape: (7, 7)
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬────────────┬─────────────┐
│ product_opti ┆ accelerator_ ┆ standalone_ ┆ age_definit ┆ variant_typ ┆ source_row ┆ notes       │
│ on           ┆ table        ┆ table       ┆ ion         ┆ e           ┆ ---        ┆ ---         │
│ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ i64        ┆ str         │
│ str          ┆ str          ┆ str         ┆ str         ┆ str         ┆            ┆             │
╞══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪════════════╪═════════════╡
│ Cancer_Prote ┆ CI_Cancer_Pr ┆ CI_Cancer_P ┆ ALB         ┆ New         ┆ 326 

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote ci_table_mapping.parquet
  Size:    3.47 KB
  SHA-256: 699dc7d0cd4e9cb6...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "CI product option → rating table mappings. 7 product options × 4 "
        "attributes (accelerator_table, standalone_table, age_definition, "
        "variant_type). Kept WIDE (not long) because pricing VLOOKUP contract "
        "needs all 4 attributes returned at once. Cross-checked: New variant "
        "products have single unified table; Existing variant products have "
        "separate accelerator/standalone tables. ALB↔New / ANB↔Existing invariant "
        "holds for all rows."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: ci_table_mapping
